In [3]:
import torch
import torch.nn as nn

# 1. 모델 설정 (입력 차원 1, 은닉 차원 1)
lstm = nn.LSTM(input_size=1, hidden_size=1, batch_first=True)

# 2. 가중치 수동 설정 (실험과 동일하게 0.5로 설정)
with torch.no_grad():
    lstm.weight_ih_l0.fill_(0.5)
    lstm.weight_hh_l0.fill_(0.5)
    lstm.bias_ih_l0.fill_(0.0)
    lstm.bias_hh_l0.fill_(0.0)

# 3. 입력 데이터 준비 (x_t=1.0, h_t-1=0.5, c_t-1=0.8)
x_t = torch.tensor([[[1.0]]]) # (batch, seq, feature)
h_0 = torch.tensor([[[0.5]]]) # (num_layers, batch, hidden)
c_0 = torch.tensor([[[0.8]]]) # (num_layers, batch, hidden)

# 4. 계산 수행
output, (h_t, c_t) = lstm(x_t, (h_0, c_0))

print(f"업데이트된 Cell State (C_t): {c_t.item():.4f}")
print(f"업데이트된 Hidden State (h_t): {h_t.item():.4f}")

업데이트된 Cell State (C_t): 0.9747
업데이트된 Hidden State (h_t): 0.5099


In [7]:
# 한글
from konlpy.tag import Okt
okt = Okt()

import pandas as pd
import re
df = pd.read_csv(r'C:\git_test\01_tuteur guide\NPL\daum_movie_review.csv')
df['target'] = df['rating'].apply(lambda x : 1 if x>0.5 else 0)
df['clean'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )

In [ ]:
df['clean'][0]

In [ ]:
okt = Okt()
def kor_tokenizer(text):    
    return [
        word for word, pos in okt.pos(text,stem=True) 
            if pos in ['Noun','Verb','Adjective'] and len(word) >=2        
           ]

In [ ]:
kor_tokenizer(df['clean'][0]),kor_tokenizer(df['clean'][1])

In [ ]:
# 단어사전 구축
from collections import Counter
vocab = Counter()
for text in df['clean']:
    vocab.update(kor_tokenizer(text))

In [ ]:
# 10000개의 단어로만 구성
vocab_size = 10000
word_to_index = {
    word:idx+2 for idx, (word,seq) in enumerate(vocab.most_common(vocab_size))
}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

def word2Sequence(text):
    return [word_to_index.get(word,1) for word in kor_tokenizer(text)]   

In [ ]:
kor_tokenizer(df['clean'][0])

In [ ]:
word2Sequence(df['clean'][0])

In [ ]:
index_to_word = {idx : word for word, idx in word_to_index.items()}
index_to_word.get(345)

In [ ]:
# padding
import torch
from torch.nn.utils.rnn import pad_sequence
x1 = word2Sequence(df['clean'][0])
x2 = word2Sequence(df['clean'][1])
x1_tensor = torch.LongTensor(x1)
x2_tensor = torch.LongTensor(x2)
print(x1_tensor,x2_tensor)
pad_sequence([x1_tensor,x2_tensor], batch_first=True, padding_value=0)

In [ ]:
MAX_LEN = 500
def padding(texts):
    result = []
    for text in texts:
        text = word2Sequence(text)[:MAX_LEN]
        text = torch.LongTensor(text)
        result.append(text)
    return pad_sequence(result,batch_first=True, padding_value=0)
    
padding(df['clean'])   